# Day 7: Data Visualization with Pandas

Today we'll learn how to create visualizations directly from pandas DataFrames and integrate with plotting libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Create sample data
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')
df = pd.DataFrame({
    'date': dates,
    'sales': np.random.normal(1000, 200, 365) + np.sin(np.arange(365) * 2 * np.pi / 365) * 100,
    'profit': np.random.normal(200, 50, 365),
    'category': np.random.choice(['A', 'B', 'C'], 365),
    'region': np.random.choice(['North', 'South', 'East', 'West'], 365)
})

print("Sample data:")
print(df.head())
print(f"Shape: {df.shape}")

## 1. Basic Plotting with Pandas

In [ ]:
# Line plot
plt.figure(figsize=(12, 4))
df.set_index('date')['sales'].plot(title='Daily Sales Over Time')
plt.ylabel('Sales')
plt.show()

# Multiple lines
plt.figure(figsize=(12, 4))
df.set_index('date')[['sales', 'profit']].plot(title='Sales and Profit Over Time')
plt.ylabel('Amount')
plt.show()

# Histogram
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['sales'].plot.hist(bins=30, title='Sales Distribution')
plt.xlabel('Sales')

plt.subplot(1, 2, 2)
df['profit'].plot.hist(bins=30, title='Profit Distribution', alpha=0.7)
plt.xlabel('Profit')
plt.tight_layout()
plt.show()

## 2. Different Plot Types

In [ ]:
# Bar plot
category_sales = df.groupby('category')['sales'].mean()
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
category_sales.plot.bar(title='Average Sales by Category')
plt.ylabel('Sales')

# Horizontal bar plot
plt.subplot(2, 2, 2)
category_sales.plot.barh(title='Average Sales by Category (Horizontal)')
plt.xlabel('Sales')

# Pie chart
plt.subplot(2, 2, 3)
category_sales.plot.pie(title='Sales Distribution by Category', autopct='%1.1f%%')
plt.ylabel('')

# Box plot
plt.subplot(2, 2, 4)
df.boxplot(column='sales', by='category', ax=plt.gca())
plt.title('Sales Distribution by Category')
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.show()

## 3. Scatter Plots and Correlation

In [ ]:
# Scatter plot
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df.plot.scatter(x='sales', y='profit', title='Sales vs Profit', alpha=0.6)

# Scatter plot with color coding
plt.subplot(1, 2, 2)
colors = {'A': 'red', 'B': 'blue', 'C': 'green'}
for category in df['category'].unique():
    subset = df[df['category'] == category]
    plt.scatter(subset['sales'], subset['profit'], 
               c=colors[category], label=category, alpha=0.6)
plt.xlabel('Sales')
plt.ylabel('Profit')
plt.title('Sales vs Profit by Category')
plt.legend()

plt.tight_layout()
plt.show()

# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_cols.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

## 4. Time Series Visualization

In [ ]:
# Set date as index for time series plotting
df_ts = df.set_index('date')

# Monthly aggregation
monthly_data = df_ts.resample('M').agg({
    'sales': 'sum',
    'profit': 'sum'
})

plt.figure(figsize=(15, 10))

# Daily data
plt.subplot(3, 1, 1)
df_ts['sales'].plot(title='Daily Sales', alpha=0.7)
df_ts['sales'].rolling(window=30).mean().plot(label='30-day MA', linewidth=2)
plt.legend()
plt.ylabel('Sales')

# Monthly data
plt.subplot(3, 1, 2)
monthly_data['sales'].plot(kind='bar', title='Monthly Sales')
plt.ylabel('Sales')
plt.xticks(rotation=45)

# Seasonal decomposition visualization
plt.subplot(3, 1, 3)
quarterly_data = df_ts.resample('Q').sum()
quarterly_data['sales'].plot(kind='line', marker='o', title='Quarterly Sales')
plt.ylabel('Sales')

plt.tight_layout()
plt.show()

# Area plot
plt.figure(figsize=(12, 6))
monthly_data.plot.area(title='Monthly Sales and Profit (Stacked Area)', alpha=0.7)
plt.ylabel('Amount')
plt.show()

## 5. Statistical Plots

In [ ]:
plt.figure(figsize=(15, 10))

# Box plots
plt.subplot(2, 3, 1)
df.boxplot(column='sales', by='category', ax=plt.gca())
plt.title('Sales by Category')
plt.suptitle('')

# Violin plot using seaborn
plt.subplot(2, 3, 2)
sns.violinplot(data=df, x='category', y='sales')
plt.title('Sales Distribution by Category')

# Density plot
plt.subplot(2, 3, 3)
for category in df['category'].unique():
    subset = df[df['category'] == category]
    subset['sales'].plot.density(label=category, alpha=0.7)
plt.title('Sales Density by Category')
plt.legend()
plt.xlabel('Sales')

# Hexbin plot
plt.subplot(2, 3, 4)
df.plot.hexbin(x='sales', y='profit', gridsize=20, ax=plt.gca())
plt.title('Sales vs Profit (Hexbin)')

# Andrews curves
plt.subplot(2, 3, 5)
from pandas.plotting import andrews_curves
sample_df = df.sample(100)  # Sample for clarity
andrews_curves(sample_df[['sales', 'profit', 'category']], 'category', ax=plt.gca())
plt.title('Andrews Curves')

# Parallel coordinates
plt.subplot(2, 3, 6)
from pandas.plotting import parallel_coordinates
sample_df_norm = sample_df.copy()
sample_df_norm['sales'] = (sample_df_norm['sales'] - sample_df_norm['sales'].mean()) / sample_df_norm['sales'].std()
sample_df_norm['profit'] = (sample_df_norm['profit'] - sample_df_norm['profit'].mean()) / sample_df_norm['profit'].std()
parallel_coordinates(sample_df_norm[['sales', 'profit', 'category']], 'category', ax=plt.gca())
plt.title('Parallel Coordinates')

plt.tight_layout()
plt.show()

## 6. Subplots and Multiple Visualizations

In [ ]:
# Create subplots using pandas
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Time series
df_ts['sales'].plot(ax=axes[0, 0], title='Daily Sales')
axes[0, 0].set_ylabel('Sales')

# Plot 2: Histogram
df['sales'].plot.hist(ax=axes[0, 1], bins=30, title='Sales Distribution')
axes[0, 1].set_xlabel('Sales')

# Plot 3: Bar chart
region_sales = df.groupby('region')['sales'].mean()
region_sales.plot.bar(ax=axes[1, 0], title='Average Sales by Region')
axes[1, 0].set_ylabel('Sales')
axes[1, 0].tick_params(axis='x', rotation=45)

# Plot 4: Scatter plot
df.plot.scatter(x='sales', y='profit', ax=axes[1, 1], title='Sales vs Profit', alpha=0.6)

plt.tight_layout()
plt.show()

# Using subplot method directly on DataFrame
numeric_df = df[['sales', 'profit']]
axes = numeric_df.plot(subplots=True, figsize=(12, 8), title=['Sales Over Time', 'Profit Over Time'])
plt.tight_layout()
plt.show()

## 7. Customizing Plots

In [ ]:
# Custom styling
plt.figure(figsize=(12, 6))

# Custom colors and styles
ax = df_ts['sales'].plot(
    color='steelblue',
    linewidth=1,
    alpha=0.7,
    title='Daily Sales with Custom Styling'
)

# Add rolling average
df_ts['sales'].rolling(window=30).mean().plot(
    ax=ax,
    color='red',
    linewidth=2,
    label='30-day Moving Average'
)

# Customize axes
ax.set_ylabel('Sales ($)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend()

# Add annotations
max_sales_date = df_ts['sales'].idxmax()
max_sales_value = df_ts['sales'].max()
ax.annotate(f'Peak: ${max_sales_value:.0f}',
           xy=(max_sales_date, max_sales_value),
           xytext=(10, 10), textcoords='offset points',
           bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
           arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.tight_layout()
plt.show()

# Custom color maps
plt.figure(figsize=(10, 6))
pivot_data = df.pivot_table(values='sales', index='region', columns='category', aggfunc='mean')
sns.heatmap(pivot_data, annot=True, cmap='viridis', fmt='.0f')
plt.title('Average Sales by Region and Category')
plt.show()

## 8. Interactive Plotting

In [ ]:
# Plotly integration (if available)
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    # Interactive line plot
    fig = px.line(df, x='date', y='sales', title='Interactive Daily Sales')
    fig.show()
    
    # Interactive scatter plot
    fig = px.scatter(df, x='sales', y='profit', color='category', 
                    title='Interactive Sales vs Profit by Category',
                    hover_data=['region'])
    fig.show()
    
    # Interactive bar chart
    category_region = df.groupby(['category', 'region'])['sales'].mean().reset_index()
    fig = px.bar(category_region, x='category', y='sales', color='region',
                title='Average Sales by Category and Region')
    fig.show()
    
except ImportError:
    print("Plotly not available. Install with: pip install plotly")
    
    # Alternative: matplotlib with widgets
    from matplotlib.widgets import Slider
    
    fig, ax = plt.subplots(figsize=(12, 6))
    plt.subplots_adjust(bottom=0.25)
    
    # Initial plot
    window_size = 30
    line, = ax.plot(df_ts.index, df_ts['sales'].rolling(window=window_size).mean())
    ax.set_title('Sales with Adjustable Moving Average')
    ax.set_ylabel('Sales')
    
    # Add slider
    ax_slider = plt.axes([0.2, 0.1, 0.5, 0.03])
    slider = Slider(ax_slider, 'Window Size', 1, 90, valinit=window_size, valfmt='%d')
    
    def update(val):
        window = int(slider.val)
        line.set_ydata(df_ts['sales'].rolling(window=window).mean())
        fig.canvas.draw()
    
    slider.on_changed(update)
    plt.show()

## 9. Advanced Visualization Techniques

In [ ]:
# Facet grids with seaborn
plt.figure(figsize=(15, 10))

# Create monthly data for faceting
df_monthly = df.copy()
df_monthly['month'] = df_monthly['date'].dt.month
df_monthly['quarter'] = df_monthly['date'].dt.quarter

# Facet grid
g = sns.FacetGrid(df_monthly, col='category', row='region', margin_titles=True, height=3)
g.map(plt.scatter, 'sales', 'profit', alpha=0.6)
g.add_legend()
plt.show()

# Pair plot
numeric_cols = ['sales', 'profit']
sample_df = df.sample(200)  # Sample for performance
g = sns.pairplot(sample_df[numeric_cols + ['category']], hue='category', diag_kind='hist')
plt.show()

# Joint plot
g = sns.jointplot(data=sample_df, x='sales', y='profit', hue='category', kind='scatter')
plt.show()

# Distribution plots
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(data=df, x='sales', hue='category', multiple='stack')
plt.title('Stacked Histogram')

plt.subplot(1, 3, 2)
sns.boxplot(data=df, x='category', y='sales')
plt.title('Box Plot by Category')

plt.subplot(1, 3, 3)
sns.violinplot(data=df, x='category', y='sales')
plt.title('Violin Plot by Category')

plt.tight_layout()
plt.show()

## 10. Saving and Exporting Plots

In [ ]:
# Create a comprehensive dashboard
fig = plt.figure(figsize=(16, 12))

# Main time series plot
ax1 = plt.subplot(3, 3, (1, 3))
df_ts['sales'].plot(ax=ax1, title='Daily Sales Over Time', color='steelblue')
df_ts['sales'].rolling(30).mean().plot(ax=ax1, color='red', label='30-day MA')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Sales distribution
ax2 = plt.subplot(3, 3, 4)
df['sales'].plot.hist(ax=ax2, bins=30, title='Sales Distribution', alpha=0.7)
ax2.axvline(df['sales'].mean(), color='red', linestyle='--', label='Mean')
ax2.legend()

# Category breakdown
ax3 = plt.subplot(3, 3, 5)
category_sales = df.groupby('category')['sales'].mean()
category_sales.plot.bar(ax=ax3, title='Avg Sales by Category')
ax3.tick_params(axis='x', rotation=0)

# Region breakdown
ax4 = plt.subplot(3, 3, 6)
region_sales = df.groupby('region')['sales'].mean()
region_sales.plot.pie(ax=ax4, title='Sales by Region', autopct='%1.1f%%')
ax4.set_ylabel('')

# Correlation heatmap
ax5 = plt.subplot(3, 3, 7)
corr_matrix = df[['sales', 'profit']].corr()
sns.heatmap(corr_matrix, annot=True, ax=ax5, cmap='coolwarm', center=0)
ax5.set_title('Correlation Matrix')

# Scatter plot
ax6 = plt.subplot(3, 3, 8)
df.plot.scatter(x='sales', y='profit', ax=ax6, alpha=0.6, title='Sales vs Profit')

# Monthly trend
ax7 = plt.subplot(3, 3, 9)
monthly_sales = df.groupby(df['date'].dt.month)['sales'].mean()
monthly_sales.plot.line(ax=ax7, marker='o', title='Monthly Sales Trend')
ax7.set_xlabel('Month')

plt.suptitle('Sales Dashboard', fontsize=16, y=0.98)
plt.tight_layout()

# Save in different formats
plt.savefig('sales_dashboard.png', dpi=300, bbox_inches='tight')
plt.savefig('sales_dashboard.pdf', bbox_inches='tight')
plt.savefig('sales_dashboard.svg', bbox_inches='tight')

print("Dashboard saved in PNG, PDF, and SVG formats")
plt.show()

# Save individual plots
fig, ax = plt.subplots(figsize=(10, 6))
df_ts['sales'].plot(ax=ax, title='Daily Sales', color='steelblue')
ax.grid(True, alpha=0.3)
plt.savefig('daily_sales.png', dpi=150, bbox_inches='tight')
plt.show()

print("Individual plot saved as daily_sales.png")

## Practice Exercises

In [ ]:
# Exercise 1: Create a comprehensive sales analysis visualization
def create_sales_analysis(df):
    """Create a comprehensive sales analysis dashboard"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Time series with trend
    df_ts = df.set_index('date')
    df_ts['sales'].plot(ax=axes[0, 0], alpha=0.7, title='Sales Over Time')
    df_ts['sales'].rolling(30).mean().plot(ax=axes[0, 0], color='red', linewidth=2)
    axes[0, 0].legend(['Daily Sales', '30-day Average'])
    
    # 2. Distribution by category
    sns.boxplot(data=df, x='category', y='sales', ax=axes[0, 1])
    axes[0, 1].set_title('Sales Distribution by Category')
    
    # 3. Regional performance
    region_stats = df.groupby('region').agg({
        'sales': ['mean', 'std'],
        'profit': 'mean'
    }).round(2)
    region_stats.columns = ['Avg_Sales', 'Sales_Std', 'Avg_Profit']
    region_stats['Avg_Sales'].plot.bar(ax=axes[0, 2], title='Average Sales by Region')
    
    # 4. Correlation analysis
    df.plot.scatter(x='sales', y='profit', ax=axes[1, 0], alpha=0.6, title='Sales vs Profit')
    
    # 5. Seasonal patterns
    df['month'] = df['date'].dt.month
    monthly_avg = df.groupby('month')['sales'].mean()
    monthly_avg.plot.line(ax=axes[1, 1], marker='o', title='Seasonal Sales Pattern')
    axes[1, 1].set_xlabel('Month')
    
    # 6. Performance heatmap
    pivot_data = df.pivot_table(values='sales', index='region', columns='category', aggfunc='mean')
    sns.heatmap(pivot_data, annot=True, ax=axes[1, 2], cmap='YlOrRd', fmt='.0f')
    axes[1, 2].set_title('Sales Heatmap: Region vs Category')
    
    plt.tight_layout()
    return fig

# Create the analysis
analysis_fig = create_sales_analysis(df)
plt.show()

# Exercise 2: Interactive time series analysis
def plot_rolling_analysis(df, windows=[7, 30, 90]):
    """Plot sales with multiple rolling averages"""
    plt.figure(figsize=(14, 8))
    
    df_ts = df.set_index('date')
    
    # Plot original data
    plt.plot(df_ts.index, df_ts['sales'], alpha=0.3, color='gray', label='Daily Sales')
    
    # Plot rolling averages
    colors = ['blue', 'red', 'green']
    for i, window in enumerate(windows):
        rolling_avg = df_ts['sales'].rolling(window=window).mean()
        plt.plot(df_ts.index, rolling_avg, color=colors[i], 
                linewidth=2, label=f'{window}-day Average')
    
    plt.title('Sales with Multiple Rolling Averages')
    plt.xlabel('Date')
    plt.ylabel('Sales')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_rolling_analysis(df)

# Exercise 3: Statistical visualization summary
def statistical_summary(df):
    """Create statistical summary visualizations"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Q-Q plot
    from scipy import stats
    stats.probplot(df['sales'], dist="norm", plot=axes[0, 0])
    axes[0, 0].set_title('Q-Q Plot: Sales vs Normal Distribution')
    
    # Residual plot
    from sklearn.linear_model import LinearRegression
    X = df[['profit']].values
    y = df['sales'].values
    model = LinearRegression().fit(X, y)
    predictions = model.predict(X)
    residuals = y - predictions
    
    axes[0, 1].scatter(predictions, residuals, alpha=0.6)
    axes[0, 1].axhline(y=0, color='red', linestyle='--')
    axes[0, 1].set_xlabel('Predicted Sales')
    axes[0, 1].set_ylabel('Residuals')
    axes[0, 1].set_title('Residual Plot')
    
    # Distribution comparison
    for category in df['category'].unique():
        subset = df[df['category'] == category]['sales']
        axes[1, 0].hist(subset, alpha=0.7, label=category, bins=20)
    axes[1, 0].set_title('Sales Distribution by Category')
    axes[1, 0].legend()
    
    # Box plot comparison
    df.boxplot(column='sales', by='region', ax=axes[1, 1])
    axes[1, 1].set_title('Sales by Region')
    plt.suptitle('')
    
    plt.tight_layout()
    plt.show()

try:
    statistical_summary(df)
except ImportError:
    print("Some statistical plots require scipy and sklearn")
    print("Install with: pip install scipy scikit-learn")

## Tomorrow's Preview
In Day 8, we'll cover:
- Performance Optimization Techniques
- Memory Management
- Efficient Data Processing
- Profiling and Benchmarking